## Tracking of a EURUSD dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Non-optimised params
num_timesteps= 100
number_particles = 250
seed = 1 # Random seem for reproducibility
c=10
prior_covar=0.0001
meas_sig2 = 0.001

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

# Step 1: Load the CSV file
excel_file = fr"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSD_exchange_rate_tracker\EURtoUSD_Historical_ExchaNge_Rates.xlsx"  # Replace with your file path
data = pd.read_excel(excel_file)

# Step 2: Extract relevant columns
discrete_time = data['Discrete_Time']
prices = data['Price']

num_timesteps= min(num_timesteps,len(discrete_time))
discrete_time=discrete_time[:num_timesteps]
prices=prices[:num_timesteps]

# Step 3: Generate Timestamps
start_time = datetime.now().replace(microsecond=0)
timesteps = [start_time + timedelta(seconds=int(dt)) for dt in discrete_time]

# Step 4: Build the Observed Prices as noisy measurements
# Define Dummy Measurement Model
measurement_model = LinearGaussian(
    ndim_state=2,  # State vector dimensions: [price, dP/dt] for the models
    mapping=[0,],  # Map the measurement to the 'price' dimension
    noise_covar=np.diag([meas_sig2])  # Small measurement noise for 'price'
)

# Generate Measurements from Ground Truth
plottable_observations=Track()
measurements = []
for i in range(len(timesteps)):
    state_vector = np.atleast_2d(prices[i]) # Price as the first axis
    timestamp = timesteps[i]
    state= GroundTruthState(state_vector=state_vector,timestamp=timestamp)
    plottable_observations.append(state)
    measurements.append(Detection(
        state_vector=state_vector,
        timestamp=timestamp,
        measurement_model=measurement_model
    ))

In [ ]:
## Priors
# Sample from the prior Gaussian distribution
from stonesoup.types.array import StateVector


# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(
    mean=np.array([prices[0],0]),  # Initial state: [price, dP/dt]
    cov=np.diag([prior_covar, prior_covar]),  # Covariance for the initial state
    size=number_particles
)

# Define covariance for particles
covars = np.stack(
    [np.eye(2) * prior_covar for _ in range(number_particles)], axis=2
)  # Shape: (2, 2, number_particles)

# Create prior particle state
lp_prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),  # Transpose states to shape (2, N)
    covariance=covars,  # Covariance matrix
    weight=np.array([Probability(1 / number_particles)] * number_particles),
    timestamp=start_time-timedelta(seconds=1)
)

gp_prior=GaussianState(state_vector=np.array([prices[0],0]),
                        covar=np.diag([prior_covar,prior_covar]),
                        timestamp=start_time-timedelta(seconds=1))

In [ ]:
## Outlining the parameter grids over which we'll estimate
# LP parameters
alpha_values = np.linspace(1.5, 1.9, 2)              
sigma_values = np.logspace(-8, -7, num=2)        
mu_values = np.linspace(-0.0001, 0.0001, 2)                
theta_values = np.linspace(0.01, 0.15, 2)                        

# GP parameters
q_values = np.logspace(-8, -7, num=20)               

In [ ]:
## Generate the necessary dicts for all the components to save time in the loop
# LP component dicts
lp_predictors= {}
resampler = SystematicResampler()

# GP component dicts
gp_predictors = {}

#Generate all the Levy process predictors and updaters
for mu_W in mu_values:
    for theta in theta_values:
        for alpha in alpha_values:
                for sigma_W2 in sigma_values:
                        lp_driver = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=NoiseCase(2))
                        lp_transition_model = LevyLangevin(driver=lp_driver, damping_coeff=theta, mu_W=mu_W)
                        lp_predictor = MarginalisedParticlePredictor(transition_model=lp_transition_model)
                        lp_predictors[(mu_W, theta, alpha, sigma_W2)] = lp_predictor

#Generate all the Gaussian process predictors and updaters
for q_val in q_values:
    gp_transition_model = ConstantVelocity(noise_diff_coeff=q_val)
    gp_predictor = KalmanPredictor(gp_transition_model)
    gp_predictors[q_val] = gp_predictor

# Build the measurement model
measurement_model = LinearGaussian(
    ndim_state=2,         # e.g. [price, dPrice/dt]
    mapping=[0,],         # measurement reads the 'price' dimension
    noise_covar=np.diag([meas_sig2])
)

# Build the updaters that depend on measurement model only
lp_updater = MarginalisedParticleUpdater(measurement_model, resampler)
gp_updater = KalmanUpdater(measurement_model)        

In [ ]:
## Filtering and likelihood calculation
from scipy.special import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

#Track dicts
lp_tracks = {}
gp_tracks = {}

#Likelihood dicts
lp_likelihoods = {}
gp_likelihoods = {}

for i, meas in enumerate(measurements):
    # Possibly you'd also keep track of prior states for each param combination
    # For each alpha, etc.:
    for mu_W in mu_values:
        for theta in theta_values:
            for alpha in alpha_values:
                for sigmaW2 in sigma_values:
                    if (mu_W, theta, alpha, sigmaW2) not in lp_likelihoods:
                        lp_tracks[(mu_W, theta, alpha, sigmaW2)]=Track()
                        prior=lp_prior
                        lp_likelihoods[(mu_W, theta, alpha, sigmaW2)] = 0.0
                    else:
                        lp_tracks[(mu_W, theta, alpha, sigmaW2)]
                        prior = lp_tracks[(mu_W, theta, alpha, sigmaW2)][-1]
                        
                    lp_predictor = lp_predictors[(mu_W, theta, alpha, sigmaW2)]
                    lp_prediction = lp_predictor.predict(prior, timestamp=meas.timestamp)
                    lp_hypothesis = SingleHypothesis(lp_prediction, meas)
                    lp_post = lp_updater.update(lp_hypothesis)
                    lp_tracks[(mu_W, theta, alpha, sigmaW2)].append(lp_post)
                    # Accumulate log-likelihood
                    lp_likelihoods[(mu_W, theta, alpha, sigmaW2)] += logsumexp(lp_updater.measurement_model.logpdf(meas,lp_post)-np.log(number_particles))
    # For Gaussian CV:
    for q_val in q_values:
        if (q_val) not in gp_likelihoods:
            gp_tracks[(q_val)]=Track()
            prior=gp_prior
            gp_likelihoods[(q_val)] = 0.0
        else:
            gp_tracks[(q_val)]
            prior = gp_tracks[(q_val)][-1]

        gp_predictor = gp_predictors[q_val]
        gp_prediction = gp_predictor.predict(prior, timestamp=meas.timestamp)
        gp_hypothesis = SingleHypothesis(gp_prediction, meas)
        gp_post = gp_updater.update(gp_hypothesis)
        gp_tracks[(q_val)].append(gp_post)
        
        gp_likelihoods[(q_val)] += gp_updater.measurement_model.logpdf(meas,gp_post)

    print(f"measurement {i} of {len(measurements)}")

In [ ]:
def summarize_top_likelihoods(lp_likelihoods, gp_likelihoods, top_n=5):
    """
    Summarize and print the top-N parameter configurations with the highest 
    log-likelihood for both the Lévy process and Gaussian process models.
    
    Parameters
    ----------
    lp_likelihoods : dict
        Dictionary keyed by (mu_W, theta, alpha, sigmaW2, meas_sig2),
        with values = total (log) likelihood.
    gp_likelihoods : dict
        Dictionary keyed by (q_val, meas_sig2), with values = total (log) likelihood.
    top_n : int, optional
        Number of highest-likelihood entries to show for each model. Default=5.
    
    Returns
    -------
    list_of_top_lp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Lévy model.
    list_of_top_gp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Gaussian process model.
    """

    # --- 1) Sort the Lévy model likelihoods ---
    #   lp_likelihoods is keyed by (mu_W, theta, alpha, sigmaW2, meas_sig2)
    #   The value is the total log-likelihood
    # We'll get items as ((mu_W, theta, alpha, sigmaW2, meas_sig2), loglike)
    list_of_lp = list(lp_likelihoods.items())
    # Sort descending by log-likelihood
    list_of_lp.sort(key=lambda x: x[1], reverse=True)
    # Take top_n
    list_of_top_lp = list_of_lp[:top_n]

    # --- 2) Sort the Gaussian process likelihoods ---
    #   gp_likelihoods is keyed by (q_val, meas_sig2)
    list_of_gp = list(gp_likelihoods.items())
    list_of_gp.sort(key=lambda x: x[1], reverse=True)
    list_of_top_gp = list_of_gp[:top_n]

    # --- 3) Print summary in a neat format ---
    print("=== Lévy Process - Top {} Log-Likelihoods ===".format(top_n))
    for rank, ((mu_W, theta, alpha, sigmaW2), loglike) in enumerate(list_of_top_lp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | mu={mu_W}, theta={theta}, alpha={alpha},"
              f" sigmaW2={sigmaW2}")

    print("")
    print("=== Gaussian CV - Top {} Log-Likelihoods ===".format(top_n))
    for rank, ((q_val), loglike) in enumerate(list_of_top_gp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | q={q_val}")

    return list_of_top_lp, list_of_top_gp

list_of_top_lp, list_of_top_gp = summarize_top_likelihoods(lp_likelihoods,gp_likelihoods,top_n=500)

In [ ]:
## Path to save plots in
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"

In [ ]:
## Parameters for tracks and plotting
LP_track=lp_tracks[(0.0001, 0.01, 1.9, 1e-8)]
GP_track=gp_tracks[(1e-8)]

label="Price"
particle_plotter_dict = {}
uncertainty=True
particle=False
plot_particle_paths=False

In [ ]:
plot_smooth=True

In [ ]:
## Plotting
file_path = Path(folder_path + rf"\TrackingPlot.html")
file_path.parent.mkdir(parents=True, exist_ok=True)
i=0
particle_plotter_dict[label]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.ONE, axis_labels=[label])
if label =="Price":
    particle_plotter_dict[label].plot_ground_truths(plottable_observations, [i], truths_label="Price Observations")

particle_plotter_dict[label].plot_tracks(LP_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Levy Process",line=dict(width=1))
particle_plotter_dict[label].plot_tracks(GP_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Constant Velocity",line=dict(width=1))
if plot_smooth is True:
    from stonesoup.smoother.particle import CarterKohnSmoother, MarginalisedKalmanSmoother, ParticleSmoother
    particlesmoother=ParticleSmoother()
    culled_track=particlesmoother.particle_paths(track=LP_track)

    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_track=RTSsmoother.smooth(track=LP_track)

    CKsmoother=CarterKohnSmoother()
    CK_track=CKsmoother.smooth(track=LP_track)
    particle_plotter_dict[label].plot_tracks(culled_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="culled",line=dict(width=1))
    particle_plotter_dict[label].plot_tracks(RTS_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="RTS",line=dict(width=1))
    particle_plotter_dict[label].plot_tracks(CK_track, [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="CK",line=dict(width=1))

particle_plotter_dict[label].fig.update_layout( 
    plot_bgcolor="white",  # Set background color to white
    xaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text="Time", font=dict(size=20)),  # Add large label
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="gray",      # Keep the grid
        title=dict(text=label, font=dict(size=20)),  # Add large label
    ),
    legend=dict(
        font=dict(size=15),       # Make the legend font larger
        # orientation='v',
        # xanchor="auto",         # Center the legend
        # yanchor="auto",           # Align the legend to the bottom of the plot
        bordercolor="Black",
        borderwidth=3,
        # y=+0.45,                   # Position it above the graph
        # x=0.6                    # Center it horizontally
    ),
)
particle_plotter_dict[label].fig.write_html(str(file_path))
particle_plotter_dict[label].fig.show()

In [ ]:
tracking_filters = ["CV", 
                    "Levy_filtered", 
                    "Levy_RTS", 
                    "Levy_CK"                               ]

from stonesoup.metricgenerator.ospametric import OSPAMetric

ospa_generators = [OSPAMetric(c=40, p=1,
                            generator_name=f'{tracking_filter} OSPA metrics',
                            tracks_key=f'tracks_{tracking_filter}',
                            truths_key='truths'
                            )
                for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean

siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                            velocity_measure=Euclidean((1, 3)),
                            generator_name=f'{tracking_filter} SIAP metrics',
                            tracks_key=f'tracks_{tracking_filter}',
                            truths_key='truths'
                            )
                for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

uncertainty_generators = [
    SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                            tracks_key=f'tracks_{tracking_filter}')
    for tracking_filter in tracking_filters]

from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager

associator = TrackToTruth(association_threshold=30)

generators = ospa_generators #+ siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data({'truths': [plottable_observations],
                        'tracks_CV': [GP_track],
                        'tracks_Levy_filtered': [LP_track],
                        'tracks_Levy_RTS': [RTS_track],
                        'tracks_Levy_CK': [CK_track]
                        })  
metrics = metric_manager.generate_metrics()

from stonesoup.plotter import MetricPlotter

# sum up distance error from ground truth over all timestamps
for tracking_filter in tracking_filters:
    total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
    print(f'OSPA total value for {tracking_filter} is {total:.3f}')

fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=['OSPA distances'])